# Data Preprocessing: Craigslist Used Cars

Raw data: 426,880 postings × 26 columns. Goal: a clean table ready for analysis. Thresholds are set once in the first code cell.

## 1. Load data

Limits chosen from the raw-data profile:
- **Price:** $500–$150,000 (32,895 prices are 0, about 9,000 more are $1–$499, and the max is $3.7 billion).
- **Year:** 1950–2022 (1900 is a placeholder; 2022 is valid for 2021 listings).
- **Odometer:** up to 500,000 miles (99th percentile is 280k; the max is 10,000,000).

In [15]:
import glob
import numpy as np
import pandas as pd

PRICE_MIN, PRICE_MAX = 500, 150_000
YEAR_MIN, YEAR_MAX = 1950, 2022
ODO_MAX = 500_000

path = (glob.glob("/kaggle/input/**/vehicles.csv", recursive=True) or glob.glob("vehicles.csv"))[0]
df = pd.read_csv(path)
raw_rows = len(df)
print("Shape:", df.shape)

Shape: (426880, 26)


## 2. Drop unused columns and clean text

- Dropped: `id` (a row number added at scraping time, so it can never identify a duplicate), `url`, `region_url`, `image_url`, `description`, `county` (100% missing), `size` (72% missing), and `lat`/`long` (`state` and `region` already give location).
- **`VIN` is kept for now** because it is needed to find duplicates in step 4.
- Text is stripped and lowercased (`model` has about 1,091 case-only variants). `VIN` is uppercased.
- `posting_date` is converted to datetime.

In [16]:
df = df.drop(columns=["id", "url", "region_url", "image_url", "description",
                      "county", "size", "lat", "long"])

text_cols = ["region", "manufacturer", "model", "condition", "cylinders", "fuel",
             "title_status", "transmission", "drive", "type", "paint_color", "state"]
for c in text_cols:
    df[c] = df[c].astype("object").str.strip().str.lower().replace("", np.nan)

df["VIN"] = df["VIN"].astype("object").str.strip().str.upper()
df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)

## 3. Invalid values

Domain limits are used instead of IQR, because price and odometer are very right-skewed and IQR would remove thousands of normal listings.

Placeholder numbers such as `11111`, `123456` and `9999999` are also removed. An odometer of 0 is kept only for new cars.

In [17]:
def is_placeholder(x):
    if pd.isna(x):
        return False
    s = str(int(x))
    return len(s) >= 5 and (len(set(s)) == 1
                            or s in "01234567890123456789"
                            or s in "98765432109876543210")

is_new = (df["condition"] == "new") | (df["year"] >= 2021)

bad = (
    (df["price"] < PRICE_MIN) | (df["price"] > PRICE_MAX) | df["price"].map(is_placeholder)
    | (df["year"].notna() & ~df["year"].between(YEAR_MIN, YEAR_MAX))
    | (df["odometer"] > ODO_MAX) | df["odometer"].map(is_placeholder)
    | ((df["odometer"] == 0) & ~is_new)
)
print("Invalid rows removed:", bad.sum())
df = df[~bad].copy()

Invalid rows removed: 46270


## 4. Duplicates

A duplicate here is **the same vehicle posted again** (reposted, or listed in several cities). It is not a repeated `id`.

- **Rule 1, VIN:** the same valid VIN means the same vehicle. A valid VIN has 17 characters from A–Z/0–9 (no I, O, Q) and is not a placeholder like `00000000000000000`. About 147k VINs repeat in the raw data.
- **Rule 2, no valid VIN:** the same manufacturer, model, year, odometer, price, state and the other vehicle attributes means the same vehicle. Only rows where those key fields are all filled are compared.
- **Keep the newest posting** (latest `posting_date`), because its price is the most current.

This runs **before** missing values are filled with `"unknown"`. Otherwise, different vehicles with unknown attributes would look identical.

In [18]:
# oldest first, so keep="last" keeps the newest posting
df = df.sort_values("posting_date", na_position="first")

# Rule 1: same valid VIN
vin = df["VIN"].fillna("")
valid_vin = vin.str.fullmatch(r"[A-HJ-NPR-Z0-9]{17}") & (vin.map(lambda v: len(set(v))) > 3)
dup_vin = df.loc[valid_vin, "VIN"].duplicated(keep="last").reindex(df.index, fill_value=False)

# Rule 2: no valid VIN -> same vehicle attributes
key = ["manufacturer", "model", "year", "odometer", "price", "state",
       "condition", "cylinders", "fuel", "transmission", "drive", "type", "paint_color"]
can_match = ~valid_vin & df[["manufacturer", "model", "year", "odometer", "price"]].notna().all(axis=1)
dup_key = df.loc[can_match].duplicated(subset=key, keep="last").reindex(df.index, fill_value=False)

print("Duplicates by VIN:", dup_vin.sum())
print("Duplicates by attributes:", dup_key.sum())

df = df[~(dup_vin | dup_key)].drop(columns="VIN")

Duplicates by VIN: 131944
Duplicates by attributes: 23747


## 5. Missing values

- **Nearly-empty rows** (fewer than 6 of 12 core fields filled) are dropped. Filling them would invent a vehicle.
- **Missing `year` or `odometer`** (about 0.3% and 1% of rows) are dropped instead of median-filled.
- **Missing categories** become `"unknown"`, which keeps the rows.

In [19]:
core = ["year", "odometer", "manufacturer", "model", "condition", "cylinders",
        "fuel", "title_status", "transmission", "drive", "type", "paint_color"]

df = df[df[core].notna().sum(axis=1) >= 6]
df = df.dropna(subset=["year", "odometer"]).copy()
df["year"] = df["year"].astype(int)
df[text_cols] = df[text_cols].fillna("unknown")

## 6. Final check

Report only, no changes.

In [20]:
df = df.reset_index(drop=True)
print(f"Final shape: {df.shape}  (kept {len(df)/raw_rows:.1%} of raw rows)")
print("Missing values:", df.isna().sum().sum())
df[["price", "year", "odometer"]].describe().round(0)

Final shape: (222569, 16)  (kept 52.1% of raw rows)
Missing values: 0


,price,year,odometer
count,222569.0,222569.0,222569.0
mean,15739.0,2009.0,106497.0
std,13847.0,10.0,65011.0
min,500.0,1950.0,0.0
25%,5999.0,2006.0,56194.0
50%,11500.0,2012.0,102383.0
75%,21000.0,2015.0,148638.0
max,150000.0,2022.0,500000.0
